# Clase 7 — Machine Learning Aplicado: Casos Reales

Hasta ahora vimos las herramientas: NumPy, Pandas, visualizaciones. Hoy las ponemos en práctica.

Vamos a trabajar con **tres casos reales** de empresas que resuelven problemas concretos con Machine Learning. En cada uno el desafío es diferente, y eso nos va a obligar a usar técnicas distintas.

Arrancamos con una aseguradora que necesita detectar fraudes entre miles de transacciones legítimas. Después pasamos a un servicio de streaming que quiere recomendar películas que le gusten al usuario. Y cerramos con una automotriz que quiere entender mejor a sus clientes para personalizar sus campañas.

Tres problemas, tres enfoques. Empecemos.

In [ ]:
# ── Instalación de librerías adicionales ───────────────────────────────────
!pip install opendatasets --quiet

import opendatasets as od
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ── Configuración visual global ────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

PALETTE = ['#7C3AED', '#0D9488', '#F59E0B', '#EF4444', '#8B5CF6', '#2DD4BF']
sns.set_palette(PALETTE)

RANDOM_STATE = 42

In [ ]:
from google.colab import userdata

# Credenciales desde Secrets de Colab (ícono de llave en el panel izquierdo)
# — más seguro que pegar el token en el código
KAGGLE_API_TOKEN = userdata.get('KAGGLE_API_TOKEN')

od.download("https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud")    # Caso 1: fraudes
od.download("https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata")    # Caso 2: películas

---
## Caso 1 — Detección de Fraudes (San Cristóbal Seguros)

Imaginá que sos parte del equipo de data science de una aseguradora. Cada día se procesan miles de transacciones con tarjeta, y una fracción muy pequeña de ellas son fraudulentas.

El problema es que esa fracción es **ridículamente pequeña**: menos del 1%. Si entrenás un modelo con esos datos sin hacer nada, el modelo va a aprender que la respuesta correcta casi siempre es "no es fraude" — y va a tener 99% de accuracy haciendo eso. Pero va a dejar pasar todos los fraudes reales.

Para resolver esto usamos **SMOTE**, una técnica que genera ejemplos sintéticos de la clase minoritaria para que el modelo pueda aprender a reconocerla. Y después entrenamos un **Random Forest** que vote entre muchos árboles de decisión para clasificar cada transacción.

In [ ]:
from sklearn.datasets import make_classification

def _datos_sinteticos_fraude(n=10000, random_state=42):
    """Plan B cuando no hay acceso a Kaggle: misma estructura y desbalance que el dataset real."""
    X, y = make_classification(
        n_samples=n, n_features=30, n_informative=5,
        n_classes=2, weights=[0.98, 0.02],
        random_state=random_state
    )
    rng = np.random.default_rng(random_state)
    df = pd.DataFrame(X, columns=[f'V{i}' for i in range(1, 31)])
    df.insert(0, 'Time', np.arange(n))
    df['Amount'] = np.abs(rng.normal(100, 50, n))
    df['Class'] = y
    return df

# El dataset real tiene 284.807 transacciones del 2013, anonimizadas con PCA.
# Solo 'Time', 'Amount' y 'Class' conservan su significado original.
try:
    df_fraud = pd.read_csv('/content/creditcardfraud/creditcard.csv')
    print(f"Dataset cargado: {df_fraud.shape}")
except Exception as e:
    print(f"Sin acceso a Kaggle ({e})\nUsando datos sintéticos...")
    df_fraud = _datos_sinteticos_fraude(random_state=RANDOM_STATE)
    print(f"Dataset sintético generado: {df_fraud.shape}")

df_fraud.shape

In [ ]:
df_fraud.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [ ]:
# ESTE ES EL PROBLEMA CENTRAL del Caso 1.
# Si entrenamos un modelo con este desbalance sin corregir, aprenderá a predecir
# "no fraude" para todo y tendrá ~99.8% de accuracy... pero 0% de utilidad real.

conteo = df_fraud['Class'].value_counts()
print(f"No fraude: {conteo[0]:,} | Fraude: {conteo[1]:,} ({conteo[1]/len(df_fraud)*100:.2f}%)")

fig, ax = plt.subplots()
conteo.plot(kind='bar', color=['#0D9488', '#EF4444'], edgecolor='white', ax=ax)
ax.set_title('Distribución de Clases — Dataset de Fraudes')
ax.set_xlabel('Clase')
ax.set_ylabel('Cantidad de transacciones')
ax.set_xticklabels(['Normal', 'Fraude'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X = df_fraud.drop('Class', axis=1)
y = df_fraud['Class']

# REGLA DE ORO: SMOTE solo se aplica al train set, nunca al test.
# Si balanceamos el test set evaluamos con datos artificiales → métricas optimistas.
# stratify=y preserva la misma proporción de fraudes en ambos conjuntos.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas")
print(f"Fraudes en train: {y_train.sum()}  |  Fraudes en test: {y_test.sum()}")

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

# Usamos el Pipeline de imbalanced-learn, que garantiza que SMOTE
# solo corra durante fit() — nunca durante predict().
# SMOTE interpola entre fraudes existentes para crear nuevos puntos sintéticos,
# así el modelo aprende una región más amplia del espacio de fraudes.

pipeline = Pipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('rf',    RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1))
])

pipeline.fit(X_train, y_train)     # SMOTE balancea → RF entrena
y_pred = pipeline.predict(X_test)  # solo RF predice, sin SMOTE

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Recall es la métrica prioritaria en fraudes:
#   Falso Negativo (FN) = fraude sin detectar → pérdida económica directa
#   Falso Positivo (FP) = alerta falsa         → molestia al cliente, resoluble con una llamada
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraude']))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Normal', 'Fraude'],
            yticklabels=['Normal', 'Fraude'], ax=ax)
ax.set_title('Matriz de Confusión — Detección de Fraudes')
ax.set_xlabel('Predicción del modelo')
ax.set_ylabel('Valor real')
plt.tight_layout()
plt.show()

In [ ]:
# predict_proba[:, 1] = probabilidad de ser fraude para cada transacción del test set.
# Usamos probabilidades (no 0/1) para poder trazar la curva ROC con todos los umbrales posibles.
# AUC = 1.0 → modelo perfecto | AUC = 0.5 → equivalente a lanzar una moneda
y_proba = pipeline.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='#7C3AED', lw=2, label=f'Modelo RF  (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='#94A3B8', lw=1, linestyle='--', label='Clasificador aleatorio (AUC = 0.5)')
ax.set_title('Curva ROC — Detección de Fraudes')
ax.set_xlabel('Tasa de Falsos Positivos (FPR)')
ax.set_ylabel('Recall — Tasa de Verdaderos Positivos (TPR)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### ¿Qué aprendimos en el Caso 1?

El desbalance era tan extremo que accuracy como métrica no servía de nada. Un modelo que siempre dice "no fraude" tiene 99% de accuracy y detecta cero fraudes.

SMOTE resolvió eso sin duplicar filas: generó puntos nuevos interpolando entre fraudes existentes, lo que le dio al modelo más información sobre cómo se ve un fraude. Y el AUC de la curva ROC nos permitió evaluar el modelo de forma honesta, independientemente del umbral de decisión que elijamos.

---
## Caso 2 — Recomendación de Películas (Amazon Prime Video)

¿Alguna vez te preguntaste cómo Netflix o Prime saben qué recomendarte después de que terminás una película?

Hay varias formas de hacerlo. Una de las más simples — y más efectivas — es mirar el contenido de las películas en sí: los géneros, las palabras clave, la temática. Si te gustó una película de ciencia ficción espacial, probablemente te interese otra que tenga esas mismas características.

Eso es exactamente lo que vamos a hacer. Vamos a convertir las características de cada película en vectores numéricos usando **TF-IDF**, y después vamos a medir qué tan "parecidos" son esos vectores usando **similitud coseno**. Las películas con vectores más parecidos son las recomendaciones.

In [ ]:
# ── Caso 2: dataset de películas TMDB ────────────────────────────────────

In [ ]:
import ast

def _dataset_peliculas_ejemplo():
    """30 películas conocidas con géneros y keywords — fallback sin Kaggle."""
    titulos = [
        'Toy Story', 'Jumanji', 'The Dark Knight', 'Inception', 'Titanic',
        'The Matrix', 'Forrest Gump', 'The Lion King', 'Finding Nemo', 'Shrek',
        'Avatar', 'Interstellar', 'Gladiator', 'Braveheart', 'The Avengers',
        'Iron Man', 'Spider-Man', 'Batman Begins', 'Superman', 'Wonder Woman',
        'Frozen', 'Moana', 'Coco', 'Up', 'WALL-E',
        'The Godfather', 'Pulp Fiction', 'Fight Club', 'The Shawshank Redemption', 'Goodfellas',
    ]
    generos = [
        "[{'id':16,'name':'Animation'},{'id':35,'name':'Comedy'},{'id':10751,'name':'Family'}]",
        "[{'id':12,'name':'Adventure'},{'id':14,'name':'Fantasy'},{'id':10751,'name':'Family'}]",
        "[{'id':28,'name':'Action'},{'id':80,'name':'Crime'},{'id':18,'name':'Drama'}]",
        "[{'id':28,'name':'Action'},{'id':878,'name':'Science Fiction'},{'id':12,'name':'Adventure'}]",
        "[{'id':18,'name':'Drama'},{'id':10749,'name':'Romance'}]",
        "[{'id':28,'name':'Action'},{'id':878,'name':'Science Fiction'}]",
        "[{'id':35,'name':'Comedy'},{'id':18,'name':'Drama'},{'id':10749,'name':'Romance'}]",
        "[{'id':16,'name':'Animation'},{'id':10751,'name':'Family'},{'id':18,'name':'Drama'}]",
        "[{'id':16,'name':'Animation'},{'id':10751,'name':'Family'},{'id':35,'name':'Comedy'}]",
        "[{'id':16,'name':'Animation'},{'id':35,'name':'Comedy'},{'id':14,'name':'Fantasy'}]",
        "[{'id':28,'name':'Action'},{'id':878,'name':'Science Fiction'},{'id':12,'name':'Adventure'}]",
        "[{'id':12,'name':'Adventure'},{'id':18,'name':'Drama'},{'id':878,'name':'Science Fiction'}]",
        "[{'id':28,'name':'Action'},{'id':18,'name':'Drama'},{'id':12,'name':'Adventure'}]",
        "[{'id':28,'name':'Action'},{'id':18,'name':'Drama'},{'id':36,'name':'History'}]",
        "[{'id':28,'name':'Action'},{'id':878,'name':'Science Fiction'},{'id':12,'name':'Adventure'}]",
        "[{'id':28,'name':'Action'},{'id':878,'name':'Science Fiction'},{'id':12,'name':'Adventure'}]",
        "[{'id':28,'name':'Action'},{'id':12,'name':'Adventure'},{'id':878,'name':'Science Fiction'}]",
        "[{'id':28,'name':'Action'},{'id':80,'name':'Crime'},{'id':18,'name':'Drama'}]",
        "[{'id':28,'name':'Action'},{'id':878,'name':'Science Fiction'},{'id':12,'name':'Adventure'}]",
        "[{'id':28,'name':'Action'},{'id':12,'name':'Adventure'},{'id':14,'name':'Fantasy'}]",
        "[{'id':16,'name':'Animation'},{'id':10751,'name':'Family'},{'id':35,'name':'Comedy'}]",
        "[{'id':16,'name':'Animation'},{'id':10751,'name':'Family'},{'id':35,'name':'Comedy'}]",
        "[{'id':16,'name':'Animation'},{'id':10751,'name':'Family'},{'id':18,'name':'Drama'}]",
        "[{'id':16,'name':'Animation'},{'id':35,'name':'Comedy'},{'id':12,'name':'Adventure'}]",
        "[{'id':16,'name':'Animation'},{'id':878,'name':'Science Fiction'},{'id':10751,'name':'Family'}]",
        "[{'id':18,'name':'Drama'},{'id':80,'name':'Crime'}]",
        "[{'id':18,'name':'Drama'},{'id':80,'name':'Crime'},{'id':53,'name':'Thriller'}]",
        "[{'id':18,'name':'Drama'}]",
        "[{'id':18,'name':'Drama'},{'id':80,'name':'Crime'}]",
        "[{'id':18,'name':'Drama'},{'id':80,'name':'Crime'}]",
    ]
    keywords = [
        "[{'id':1,'name':'toy'},{'id':2,'name':'animation'},{'id':3,'name':'friendship'}]",
        "[{'id':4,'name':'board game'},{'id':5,'name':'jungle'},{'id':6,'name':'adventure'}]",
        "[{'id':7,'name':'superhero'},{'id':8,'name':'villain'},{'id':9,'name':'dark'}]",
        "[{'id':10,'name':'dream'},{'id':11,'name':'mind'},{'id':12,'name':'heist'}]",
        "[{'id':13,'name':'ship'},{'id':14,'name':'love'},{'id':15,'name':'disaster'}]",
        "[{'id':16,'name':'hacker'},{'id':17,'name':'simulation'},{'id':18,'name':'future'}]",
        "[{'id':19,'name':'life'},{'id':20,'name':'history'},{'id':21,'name':'love story'}]",
        "[{'id':22,'name':'lion'},{'id':23,'name':'africa'},{'id':24,'name':'kingdom'}]",
        "[{'id':25,'name':'fish'},{'id':26,'name':'ocean'},{'id':27,'name':'friendship'}]",
        "[{'id':28,'name':'ogre'},{'id':29,'name':'fairy tale'},{'id':30,'name':'comedy'}]",
        "[{'id':31,'name':'alien'},{'id':32,'name':'planet'},{'id':33,'name':'future'}]",
        "[{'id':34,'name':'space'},{'id':35,'name':'time'},{'id':36,'name':'love'}]",
        "[{'id':37,'name':'rome'},{'id':38,'name':'warrior'},{'id':39,'name':'revenge'}]",
        "[{'id':40,'name':'scotland'},{'id':41,'name':'war'},{'id':42,'name':'freedom'}]",
        "[{'id':43,'name':'superhero'},{'id':44,'name':'team'},{'id':45,'name':'alien'}]",
        "[{'id':46,'name':'superhero'},{'id':47,'name':'technology'},{'id':48,'name':'armor'}]",
        "[{'id':49,'name':'superhero'},{'id':50,'name':'spider'},{'id':51,'name':'power'}]",
        "[{'id':52,'name':'superhero'},{'id':53,'name':'villain'},{'id':54,'name':'fear'}]",
        "[{'id':55,'name':'superhero'},{'id':56,'name':'alien'},{'id':57,'name':'power'}]",
        "[{'id':58,'name':'superhero'},{'id':59,'name':'warrior'},{'id':60,'name':'amazon'}]",
        "[{'id':61,'name':'princess'},{'id':62,'name':'snow'},{'id':63,'name':'magic'}]",
        "[{'id':64,'name':'ocean'},{'id':65,'name':'island'},{'id':66,'name':'adventure'}]",
        "[{'id':67,'name':'music'},{'id':68,'name':'death'},{'id':69,'name':'family'}]",
        "[{'id':70,'name':'adventure'},{'id':71,'name':'old man'},{'id':72,'name':'balloon'}]",
        "[{'id':73,'name':'robot'},{'id':74,'name':'earth'},{'id':75,'name':'love'}]",
        "[{'id':76,'name':'mafia'},{'id':77,'name':'family'},{'id':78,'name':'power'}]",
        "[{'id':79,'name':'crime'},{'id':80,'name':'dark comedy'},{'id':81,'name':'violence'}]",
        "[{'id':82,'name':'identity'},{'id':83,'name':'chaos'},{'id':84,'name':'underground'}]",
        "[{'id':85,'name':'prison'},{'id':86,'name':'hope'},{'id':87,'name':'friendship'}]",
        "[{'id':88,'name':'mafia'},{'id':89,'name':'crime'},{'id':90,'name':'rise'}]",
    ]
    return pd.DataFrame({'title': titulos, 'genres': generos, 'keywords': keywords})

# Las columnas 'genres' y 'keywords' vienen como strings con formato JSON
# '[{"id": 28, "name": "Action"}, ...]' — las convertimos a texto plano más adelante.
try:
    df_movies = pd.read_csv('/content/tmdb-movie-metadata/tmdb_5000_movies.csv')
    if 'title' not in df_movies.columns or 'genres' not in df_movies.columns:
        raise ValueError("Columnas esperadas no encontradas")
    print(f"Dataset cargado: {df_movies.shape}")
except Exception as e:
    print(f"Sin acceso a Kaggle ({e})\nUsando dataset de ejemplo...")
    df_movies = _dataset_peliculas_ejemplo()
    print(f"Dataset de ejemplo: {df_movies.shape}")

df_movies.head()

In [ ]:
df_movies[['title', 'genres', 'keywords']].head(3)

,title,genres,keywords
0,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":..."
1,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na..."
2,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name..."


In [ ]:
def extraer_nombres(texto):
    """
    '[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}]'  →  'Action Adventure'

    ast.literal_eval es más seguro que eval: solo acepta literales Python, no código arbitrario.
    """
    try:
        return ' '.join(item['name'] for item in ast.literal_eval(texto))
    except (ValueError, SyntaxError):
        return ''

df_movies['genres_clean']    = df_movies['genres'].apply(extraer_nombres)
df_movies['keywords_clean']  = df_movies['keywords'].apply(extraer_nombres)

# Huella digital textual de cada película: géneros + keywords combinados
df_movies['combined_features'] = df_movies['genres_clean'] + ' ' + df_movies['keywords_clean']

df_movies[['title', 'combined_features']].head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# TF-IDF convierte cada película en un vector numérico donde cada posición es una palabra
# del vocabulario. TF penaliza palabras raras en esa película; IDF penaliza palabras que
# aparecen en muchas películas (como "Adventure") porque no ayudan a diferenciarlas.
# stop_words='english' elimina artículos y preposiciones sin significado.
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_movies['combined_features'])

# Similitud coseno: mide el ángulo entre dos vectores.
# 1.0 = películas idénticas | 0.0 = sin ninguna palabra en común
# Calculamos todas las combinaciones de una vez → matriz cuadrada (n_películas × n_películas)
similitud = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Vocabulario: {len(tfidf.vocabulary_)} palabras | Matriz de similitud: {similitud.shape}")

In [ ]:
def recomendar(titulo, n=5):
    """Devuelve las n películas más similares usando TF-IDF + similitud coseno."""
    matches = df_movies[df_movies['title'].str.contains(titulo, case=False, na=False)]
    if matches.empty:
        print(f"No se encontró '{titulo}' en el dataset.")
        return None

    idx = matches.index[0]

    # similitud[idx] = scores vs todas las películas; [1:] excluye la propia película (score=1.0)
    scores = sorted(enumerate(similitud[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    indices = [i for i, _ in scores]

    resultado = df_movies.iloc[indices][['title', 'genres_clean']].copy()
    resultado['similitud'] = [round(s, 4) for _, s in scores]
    return resultado

In [ ]:
# ── Prueba 1: Toy Story ───────────────────────────────────────────────────
# Esperamos películas de animación familiar con temáticas de amistad/aventura
print("Recomendaciones para 'Toy Story':\n")
resultado = recomendar('Toy Story')
resultado

In [ ]:
# ── Prueba 2: Avatar ──────────────────────────────────────────────────────
# Esperamos películas de acción/ciencia ficción con mundos alienígenas o espaciales
print("Recomendaciones para 'Avatar':\n")
resultado = recomendar('Avatar')
resultado

### ¿Qué aprendimos en el Caso 2?

El truco central fue convertir texto en números de forma inteligente. TF-IDF no trata todas las palabras igual: premia las palabras que son características de una película y penaliza las que aparecen en todas (como "Adventure").

Una vez que cada película es un vector, la similitud coseno hace el trabajo de comparación. No importa si una película tiene más palabras que otra — solo importa la dirección del vector, no su magnitud. Eso es lo que hace que la comparación sea justa.

---
## Caso 3 — Segmentación de Clientes (Mazda Argentina)

No todos los clientes son iguales, y tratar a todos igual es desperdiciar esfuerzo y presupuesto de marketing.

Mazda quiere entender mejor a su base de clientes: ¿quiénes son los que gastan mucho pero ganan poco (los que se estiran para comprar)? ¿y los que ganan bien pero son conservadores con el gasto? Cada perfil necesita una estrategia distinta.

El problema es que nadie nos dijo de antemano cuántos grupos hay ni cómo se llaman. Acá entra el **aprendizaje no supervisado**: le damos los datos al algoritmo y le pedimos que encuentre la estructura por sí solo. Vamos a usar **KMeans**, que agrupa puntos según su distancia en el espacio. El único parámetro que tenemos que definir es cuántos grupos queremos — y para eso usamos el **método del codo** y el **coeficiente de silueta**.

In [ ]:
np.random.seed(RANDOM_STATE)

# 4 perfiles de clientes con distribución gaussiana alrededor de su centro (media, std)
# KMeans los va a descubrir sin saber de antemano que existen
segmentos = {
    'Ahorradores':      {'ingreso': (30, 8),  'gasto': (20, 7),  'n': 125},
    'Conservadores':    {'ingreso': (70, 10), 'gasto': (25, 8),  'n': 125},
    'Jóvenes impulso':  {'ingreso': (35, 8),  'gasto': (75, 8),  'n': 125},
    'Premium':          {'ingreso': (80, 10), 'gasto': (80, 8),  'n': 125},
}

datos = []
for _, p in segmentos.items():
    ingreso = np.random.normal(p['ingreso'][0], p['ingreso'][1], p['n'])
    gasto   = np.random.normal(p['gasto'][0],   p['gasto'][1],   p['n'])
    datos.append(pd.DataFrame({'ingreso_anual_k': ingreso, 'puntaje_gasto': gasto}))

df_clientes = pd.concat(datos, ignore_index=True).clip(lower=0, upper=100)

df_clientes.describe().round(1)

In [ ]:
# Antes de aplicar cualquier algoritmo, graficamos los datos sin etiquetar.
# Si ya vemos nubes separadas a simple vista, el clustering va a funcionar bien.

fig, ax = plt.subplots()
ax.scatter(df_clientes['ingreso_anual_k'], df_clientes['puntaje_gasto'],
           color='#7C3AED', alpha=0.4, edgecolors='white', linewidths=0.3)
ax.set_title('Exploración Inicial — Clientes de Mazda')
ax.set_xlabel('Ingreso Anual (miles USD)')
ax.set_ylabel('Puntaje de Gasto (0-100)')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# KMeans usa distancias → variables en escalas distintas distorsionan los resultados.
# StandardScaler lleva cada variable a media=0 y desvío=1 para que pesen igual.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clientes)

In [ ]:
K_range = range(2, 11)
inercias, siluetas = [], []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inercias, 'o-', color='#7C3AED', linewidth=2, markersize=8)
axes[0].set_title('Método del Codo')
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inercia')

axes[1].plot(K_range, siluetas, 's-', color='#0D9488', linewidth=2, markersize=8)
axes[1].set_title('Coeficiente de Silueta')
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silueta promedio')

plt.tight_layout()
plt.show()

k_optimo = list(K_range)[siluetas.index(max(siluetas))]
print(f"K óptimo según silueta: {k_optimo}  (score = {max(siluetas):.3f})")

In [ ]:
km_final = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
df_clientes['segmento'] = km_final.fit_predict(X_scaled)

colores = ['#7C3AED', '#0D9488', '#F59E0B', '#EF4444']

fig, ax = plt.subplots(figsize=(10, 7))

for i in range(4):
    mask = df_clientes['segmento'] == i
    ax.scatter(df_clientes.loc[mask, 'ingreso_anual_k'],
               df_clientes.loc[mask, 'puntaje_gasto'],
               c=colores[i], label=f'Segmento {i}',
               alpha=0.7, edgecolors='white', linewidths=0.3, s=60)

# inverse_transform convierte los centroides de vuelta a la escala original
centroides = scaler.inverse_transform(km_final.cluster_centers_)
ax.scatter(centroides[:, 0], centroides[:, 1],
           c='white', s=200, marker='X', edgecolors='#1E1B4B', linewidths=2, zorder=5,
           label='Centroides')

ax.set_title('Segmentación de Clientes — Mazda Argentina (K=4)')
ax.set_xlabel('Ingreso Anual (miles USD)')
ax.set_ylabel('Puntaje de Gasto (0-100)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
resumen = df_clientes.groupby('segmento').agg(
    cantidad         = ('ingreso_anual_k', 'count'),
    ingreso_promedio = ('ingreso_anual_k', 'mean'),
    gasto_promedio   = ('puntaje_gasto',   'mean'),
    ingreso_std      = ('ingreso_anual_k', 'std'),
    gasto_std        = ('puntaje_gasto',   'std'),
).round(1)

# Interpretación de cada segmento según sus promedios:
#   ingreso alto + gasto alto  → Premium
#   ingreso alto + gasto bajo  → Conservadores
#   ingreso bajo + gasto alto  → Jóvenes de impulso
#   ingreso bajo + gasto bajo  → Ahorradores
print("Segmentos detectados por KMeans:\n")
resumen

### ¿Qué aprendimos en el Caso 3?

KMeans no sabe nada de los clientes antes de empezar — solo recibe coordenadas y agrupa. Lo interesante es que el algoritmo encontró exactamente los 4 perfiles que tenían sentido de negocio, sin que nadie se los dijera.

El método del codo y la silueta nos dieron confianza en la elección de K=4. En un caso real, esa elección también tiene que estar validada por alguien que conozca el negocio: los números te dicen cuántos clusters son matemáticamente razonables, pero el negocio te dice cuáles tienen sentido.

---
## Cierre de la clase

Trabajamos con tres problemas distintos y en cada uno la solución fue diferente. Eso es la idea central: no existe una técnica universal. Hay que leer el problema antes de elegir la herramienta.

| Caso | Problema | Técnica | Métrica clave |
|------|----------|---------|---------------|
| Fraudes | Clases muy desbalanceadas | SMOTE + Random Forest | Recall / AUC-ROC |
| Películas | Recomendación por contenido | TF-IDF + Similitud Coseno | Similitud (0 a 1) |
| Clientes | Segmentación sin etiquetas | KMeans | Silueta / Inercia |

Y en los tres casos, el preprocesamiento fue tan importante como el modelo en sí: balancear las clases, limpiar el texto, escalar las variables. Un modelo bien elegido con datos mal preparados va a dar malos resultados.

---
*Material preparado por el **Ing. Juan Cruz Alric** — Introducción a la Inteligencia Artificial*